In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from datetime import datetime
import csv

# Ensure output directory exists
output_dir = "experiments/fold_plots"
os.makedirs(output_dir, exist_ok=True)

# Step 1: Load and process individual JSONL timing files
def process_timings(jsonl_path, output_prefix):
    rows = []

    with open(jsonl_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            try:
                row = json.loads(line)
                rows.append({
                    "type": output_prefix,
                    "start_ms": int(row["start_ms"]),
                    "duration_ms": int(row["duration_ms"]),
                    "num_scans_per_fold": int(row["n"]),
                    "iteration": i
                })
            except json.JSONDecodeError as e:
                print(f"⚠️ Skipping invalid JSON line: {e}")

    df = pd.DataFrame(rows)
    if df.empty:
        print(f"⚠️ No timing data found in {jsonl_path}. Skipping.")
        return None

    base_path = f"experiments/{output_prefix}_timings_summary.csv"
    df.to_csv(base_path, index=False)
    print(f"✅ Saved timing summary to {base_path}")

    return df

def process_proof_sizes(jsonl_path, output_csv_path):
    rows = []

    with open(jsonl_path, 'r') as f:
        for line in f:
            try:
                row = json.loads(line)
                rows.append({
                    "proof_size": int(row["proof_size"]),
                    "payload_size": int(row["payload_size"]),
                    "n": int(row["n"])
                })
            except (json.JSONDecodeError, KeyError) as e:
                print(f"⚠️ Skipping invalid or incomplete JSON line: {e}")

    df = pd.DataFrame(rows)
    if df.empty:
        print(f"⚠️ No proof size data found in {jsonl_path}. Skipping.")
        return None

    df.sort_values("n", inplace=True)
    df.to_csv(output_csv_path, index=False)
    print(f"✅ Saved proof size summary to {output_csv_path}")
    return df


def process_latency_timings(jsonl_path, output_prefix):
    rows = []

    with open(jsonl_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            try:
                row = json.loads(line)
                rows.append({
                    "type": output_prefix,
                    "start_ms": int(row["start_ms"]),
                    "duration_ms": int(row["duration_ms"]),
                    "num_scans_per_fold": int(row["n"]),
                    "iteration": i
                })
            except (json.JSONDecodeError, KeyError) as e:
                print(f"⚠️ Skipping invalid JSON line in {output_prefix}: {e}")

    df = pd.DataFrame(rows)
    if df.empty:
        print(f"⚠️ No latency data found in {jsonl_path}. Skipping.")
        return None

    # Group by 'n' and compute average duration
    grouped = df.groupby("num_scans_per_fold")["duration_ms"].agg(
        avg_duration_ms="mean",
        std_duration_ms="std",
        num_samples="count"
    ).reset_index()

    summary_csv_path = f"experiments/{output_prefix}_latency_summary.csv"
    grouped.to_csv(summary_csv_path, index=False)
    print(f"✅ Saved latency average summary to {summary_csv_path}")

    return df

# Step 2.5: Process proof size data
proof_size_path = "json_files/fold/proof_size.jsonl"
proof_size_csv_path = "experiments/fold_proof_sizes_summary.csv"
proof_df = process_proof_sizes(proof_size_path, proof_size_csv_path)

# Step 2: Process all timing types
file_info = {
    "json_files/fold/step_timings.jsonl": "fold_step",
    "json_files/fold/latency_timings.jsonl": "fold_latency",
    "json_files/fold/verify_timings.jsonl": "fold_verify",
    "json_files/fold/proof_timings.jsonl": "fold_proof"
}

dfs = {}
for path, label in file_info.items():
    df = process_timings(path, label)
    if df is not None:
        dfs[label] = df

# Step 3: Compute average durations grouped by batch size (num_scans_per_fold)
k = 100  # total callbacks

def get_avg(df, name):
    return df.groupby("num_scans_per_fold")["duration_ms"].mean().rename(name)

avg_step = get_avg(dfs["fold_step"], "AvgStepTime")
avg_final = get_avg(dfs["fold_proof"], "AvgProofTime")
avg_total = get_avg(dfs["fold_latency"], "AvgLatencyTime")
avg_verify = get_avg(dfs["fold_verify"], "AvgVerifyTime")

# Step 4: Compute amortized unit costs
unit_costs = pd.concat([avg_step, avg_final, avg_total, avg_verify], axis=1)
unit_costs["t"] = (k // unit_costs.index).astype(int)
unit_costs["LatencyUnitCost"] = unit_costs["AvgLatencyTime"] / (unit_costs["t"] * unit_costs.index)
unit_costs["StepUnitCost"] = unit_costs["AvgStepTime"] / (unit_costs["t"] * unit_costs.index)
unit_costs["ProofUnitCost"] = unit_costs["AvgProofTime"] / (unit_costs["t"] * unit_costs.index)
unit_costs["VerifyUnitCost"] = unit_costs["AvgVerifyTime"] / (unit_costs["t"] * unit_costs.index)

unit_costs_path = "experiments/fold_unit_costs_summary.csv"
unit_costs.to_csv(unit_costs_path)
print(f"📊 Saved unit cost summary to {unit_costs_path}")

# Step 5: Plotting setup for PGF + PDF
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "text.usetex": True,
    "pgf.texsystem": "pdflatex",
    "font.family": "sans-serif",
    "font.sans-serif": "Arial",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 11,
    "figure.titlesize": 14,
    "pgf.rcfonts": False
})

def save_plot(fig, filename):
    pdf_path = os.path.join(output_dir, f"{filename}.pdf")
    pgf_path = os.path.join(output_dir, f"{filename}.pgf")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(pgf_path, bbox_inches="tight")
    print(f"✅ Saved plots as {pdf_path} and {pgf_path}")

# Color-blind friendly palette (Okabe-Ito)
okabe_ito = [
    "#E69F00", "#56B4E9", "#009E73",  "#CC79A7",
    "#F0E442", "#0072B2", "#D55E00", "#999999"
]

folding_latencies = {
    1: 9.3,
    2: 9.8,
    4: 10.0,
    5: 10.6,
    10: 12.5,
    20: 14.4,
    25: 15.9,
    50: 20.0,
}

# Scan latency line: y = 4.0 * k
k_vals = np.linspace(1, 100, 100)
scan_latency = 2.0 * k_vals

# Start plotting
plt.figure(figsize=(7, 5))

# Plot scan latency line
plt.plot(k_vals, scan_latency, label= r"Baseline Latency: $2.0 \cdot k$", color='black')

# Plot folding latencies with distinct colors
for i, (n, fold_latency) in enumerate(folding_latencies.items()):
    color = okabe_ito[i % len(okabe_ito)]
    k_intersect = fold_latency / 2.0
    plt.axhline(fold_latency, linestyle='--', color=color,
                label = rf"$(n={n})$: ${fold_latency:.1f}$ ms, $k={k_intersect:.1f}$")
    plt.plot(k_intersect, fold_latency, 'o', color=color)

# Labels and layout
plt.xlabel(r"\# Callbacks Scanned ($k$)")
plt.ylabel("Server Workload (ms)")
plt.title("Regular vs Folded Scan Server Workload Comparison on Modern Hardware")
plt.grid(True)
plt.legend(fontsize=11)
plt.tight_layout()

# Save plots
os.makedirs("experiments/fold_plots", exist_ok=True)
plt.savefig("experiments/fold_plots/scan_vs_folding_latency.pdf", bbox_inches='tight')
plt.savefig("experiments/fold_plots/scan_vs_folding_latency.pgf", bbox_inches='tight')
plt.show()

# Server-side folding analysis comparison graph for legacy hardware
folding_latencies_legacy = {
    1: 223.7,
    2: 221.4,
    4: 277.9,
    5: 276.8,
    10: 435.3,
    20: 745.2,
    25: 886.0,
    50: 3135.8,
}

# Scan latency line: y = 9.16 * k
k_vals = np.linspace(1, 350, 100)
scan_latency = 9.16 * k_vals

# Start plotting
plt.figure(figsize=(7, 5))

# Plot scan latency line
plt.plot(k_vals, scan_latency, label= r"Baseline Latency: $9.16 \cdot k$", color='black')

# Plot folding latencies with distinct colors
for i, (n, fold_latency) in enumerate(folding_latencies_legacy.items()):
    color = okabe_ito[i % len(okabe_ito)]
    k_intersect = fold_latency / 9.16
    plt.axhline(fold_latency, linestyle='--', color=color,
                label = rf"$(n={n})$: ${fold_latency:.1f}$ ms, $k={k_intersect:.1f}$")
    plt.plot(k_intersect, fold_latency, 'o', color=color)


# Labels and layout
plt.xlabel(r"\# Callbacks Scanned ($k$)")
plt.ylabel("Server Workload (ms)")
plt.title("Regular vs Folded Scan Server Workload Comparison on Legacy Hardware")
plt.grid(True)
plt.legend(fontsize=11)
plt.tight_layout()

# Save plots
os.makedirs("experiments/fold_plots", exist_ok=True)
plt.savefig("experiments/fold_plots/scan_vs_folding_latency_legacy.pdf", bbox_inches='tight')
plt.savefig("experiments/fold_plots/scan_vs_folding_latency_legacy.pgf", bbox_inches='tight')
plt.show()

# Folding client side graph for modern hardware

# Constants
lhs_coef = 210.48

# Data in the format (n, rhs_coef, intercept)
equations = [
    (1, 173.03, 426.7),
    (2, 98.15, 498.1),
    (4, 61.42, 646.4),
    (5, 54.36, 715.1),
    (10, 41.34, 1083.5),
    (20, 33.59, 1807.0),
    (25, 32.12, 2171.0),
    (50, 28.10, 3929.1),
]

# Solve for k: lhs_coef * k = rhs_coef * k + intercept
ideal_k = {}
for n, rhs_coef, intercept in equations:
    k = intercept / (lhs_coef - rhs_coef)
    ideal_k[n] = k

# Plotting
k_values = np.linspace(0, 50, 500)  # max(ideal_k.values()) * 1.1
plt.figure(figsize=(8, 7))

# Baseline
plt.plot(k_values, lhs_coef * k_values, label=r"Baseline ProofGen: $210.48 \cdot x$", color='black')

for i, (n, rhs_coef, intercept) in enumerate(equations):
    k = ideal_k[n]
    rhs_line = rhs_coef * k_values + intercept
    color = okabe_ito[i % len(okabe_ito)]
    plt.plot(k_values, rhs_line,
             label = rf"$(n={n})$: Break-even $={k:.2f}$",
             linestyle='--',
             color=color)

# Dots at ideal k values (intersections)
for i, (n, k) in enumerate(ideal_k.items()):
    color = okabe_ito[i % len(okabe_ito)]
    y = lhs_coef * k  # since this is where lhs and rhs intersect
    plt.plot([k], [y], marker='o', color=color) 


# Labels and legend
plt.xlabel(r"\# Callbacks Scanned ($x$)", fontsize=16)
plt.ylabel("Client Workload (ms)", fontsize=15)
plt.title("Regular vs Folded Scan Client Workload Comparison on Modern Hardware", fontsize=15)
plt.legend(loc='upper left', fontsize=15) 
plt.grid(True)
plt.tight_layout()

# Save plot
plot_path_pgf = "experiments/fold_plots/ideal_k_comparison.pgf"
plot_path_pdf = "experiments/fold_plots/ideal_k_comparison.pdf"
plt.savefig(plot_path_pgf, bbox_inches='tight')
plt.savefig(plot_path_pdf, bbox_inches='tight')
plt.show()
plt.close()


# Folding client side graph for legacy hardware

# Constants
lhs_coef_legacy = 739.54

# Data in the format (n, rhs_coef, intercept)
equations_legacy = [
    (1, 606.09, 1136.3),
    (2, 345.99, 1312.1),
    (4, 217.81, 1666.3),
    (5, 191.78, 1783.3),
    (10, 138.53, 2618.2),
    (20, 114.25, 4471.8),
    (25, 105.64, 5264.2),
    (50, 92.72, 9311.2),
]

# Solve for k: lhs_coef * k = rhs_coef * k + intercept
ideal_k = {}
for n, rhs_coef, intercept in equations_legacy:
    k = intercept / (lhs_coef_legacy - rhs_coef)
    ideal_k[n] = k

# Plotting
k_values = np.linspace(0, 50, 500) # max(ideal_k.values()) * 1.1
plt.figure(figsize=(8, 7))

# Baseline
plt.plot(k_values, lhs_coef_legacy * k_values, label=r"Baseline ProofGen: $739.54 \cdot x$", color='black')

for i, (n, rhs_coef, intercept) in enumerate(equations_legacy):
    k = ideal_k[n]
    rhs_line = rhs_coef * k_values + intercept
    color = okabe_ito[i % len(okabe_ito)]
    plt.plot(k_values, rhs_line,
             label = rf"$(n={n})$: Break-even $={k:.2f}$",
             linestyle='--',
             color=color)
    

# Dots at ideal k values (intersections)
for i, (n, k) in enumerate(ideal_k.items()):
    color = okabe_ito[i % len(okabe_ito)]
    y = lhs_coef_legacy * k  # since this is where lhs and rhs intersect
    plt.plot([k], [y], marker='o', color=color)


# Labels and legend
plt.xlabel(r"\# Callbacks Scanned ($x$)", fontsize=16)
plt.ylabel("Client Workload (ms)", fontsize=15)
plt.title("Regular vs Folded Scan Client Workload Comparison on Legacy Hardware", fontsize=15)
plt.legend(loc='upper left', fontsize=15)
plt.grid(True)
plt.tight_layout()

# Save plot
plot_path_pgf = "experiments/fold_plots/ideal_k_comparison_legacy.pgf"
plot_path_pdf = "experiments/fold_plots/ideal_k_comparison_legacy.pdf"
plt.savefig(plot_path_pgf, bbox_inches='tight')
plt.savefig(plot_path_pdf, bbox_inches='tight')
plt.show()
plt.close()

# Prepare a container for results
results = {
    "server_modern": {},
    "server_legacy": {},
    "client_modern": {},
    "client_legacy": {}
}

# --- Server modern ---
for n, fold_latency in folding_latencies.items():
    k_intersect = fold_latency / 4.0  # baseline slope = 4.0
    results["server_modern"][n] = {
        "fold_latency": fold_latency,
        "ideal_k": k_intersect
    }

# --- Server legacy ---
for n, fold_latency in folding_latencies_legacy.items():
    k_intersect = fold_latency / 9.16  # baseline slope = 9.16
    results["server_legacy"][n] = {
        "fold_latency": fold_latency,
        "ideal_k": k_intersect
    }

# --- Client modern ---
lhs_coef = 210.48
for n, rhs_coef, intercept in equations:
    k = intercept / (lhs_coef - rhs_coef)
    results["client_modern"][n] = {
        "rhs_coef": rhs_coef,
        "intercept": intercept,
        "ideal_k": k
    }

# --- Client legacy ---
lhs_coef_legacy = 739.54
for n, rhs_coef, intercept in equations_legacy:
    k = intercept / (lhs_coef_legacy - rhs_coef)
    results["client_legacy"][n] = {
        "rhs_coef": rhs_coef,
        "intercept": intercept,
        "ideal_k": k
    }

# --- Save to JSON ---
os.makedirs("experiments/fold_data", exist_ok=True)
with open("experiments/fold_data/ideal_k_values.json", "w") as f:
    json.dump(results, f, indent=4)

# --- Save to CSV ---
with open("experiments/fold_data/ideal_k_values.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["graph", "n", "rhs_coef", "intercept", "fold_latency", "ideal_k"])
    
    for graph, data in results.items():
        for n, vals in data.items():
            writer.writerow([
                graph,
                n,
                vals.get("rhs_coef", ""),
                vals.get("intercept", ""),
                vals.get("fold_latency", ""),
                vals["ideal_k"]
            ])